# Fine-tuning LLaMA for EV Charging Data Analysis
#### Enhancing analytical capabilities for County of Santa Barbara Zero Emission Vehicle specialists.

## Project Objective

This effort aims to fine-tune the LLaMA open-source language model using datasets such as the `nvidia/OpenMathInstruct-2` dataset. The goal is to enhance the model's ability to assist County of Santa Barbara Zero Emission Vehicle specialists in understanding and analyzing EV charging station data and related metrics.

The `nvidia/OpenMathInstruct-2` dataset, which contains mathematical instructions and problems, will be used to train the LLaMA model. While this dataset is not directly related to EV charging data, fine-tuning on a dataset with a strong focus on structured reasoning and problem-solving, like OpenMathInstruct-2, can improve the model's overall analytical capabilities. This enhanced analytical ability can then be applied to the domain of EV charging data, helping the model to better understand patterns, calculate metrics, and potentially identify insights relevant to the Zero Emission Vehicle specialists. The fine-tuned model could potentially assist with tasks such as:

* Explaining complex data analysis results related to charging station usage.
* Generating summaries of key metrics from EV charging data.
* Answering questions about trends and patterns in EV adoption and charging infrastructure.
* Potentially assisting with forecasting or scenario analysis based on historical data.

By improving the model's foundational reasoning skills with the OpenMathInstruct-2 dataset, we aim to make it a more powerful tool for the specialists working with EV charging data.

## Application / Deployment Context

The objective is to use this model as an inference endpoint in conjunction with the County of Santa Barbara AI Interface application. This application is a RAG system which leverages several models including other LLaMA models. Several documents such as the Zero-Emission Vehicle plan, the Climate Action Plan and transportation ordinances have been uploaded to the vector database. By combining vectorized knowledge base context with the problem solving capabilities of the OpenMathInstruct-2 fine tuning we are striving for a specialized AI experience tailored to the County's Sustainability Division's unique needs. Over time we might fine tune with other data sets such as ones related to California zoning, ordinances, regulations and laws. Experimentation and deliberation will be needed to determine if RAG or fine tuning creates the best results.

## Setup and Installation

Install required packages for fine-tuning LLaMA with QLoRA.

In [ ]:
!pip install -q -U transformers peft accelerate trl
!pip install -U bitsandbytes

## Dataset Preparation

Load the NVIDIA OpenMathInstruct-2 dataset and convert it to JSONL format for training.

In [ ]:
import json
from datasets import load_dataset
from tqdm import tqdm

# Load the dataset from Hugging Face
dataset = load_dataset('nvidia/OpenMathInstruct-2', split='train')

print("Converting dataset to jsonl format")
output_file = "openmathinstruct2.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for item in tqdm(dataset):
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"Conversion complete. Output saved as {output_file}")

# Reference: https://huggingface.co/datasets/nvidia/OpenMathInstruct-2

## Hugging Face Authentication

You need to authenticate with Hugging Face to access the LLaMA model.

**Steps:**
1. Go to [Hugging Face Settings](https://huggingface.co/settings/tokens)
2. Create a new access token with "read" permissions
3. Run the cell below and enter your token when prompted

**Alternative:** Set the token as an environment variable:
```bash
export HUGGING_FACE_HUB_TOKEN="your_token_here"
```

In [ ]:
from huggingface_hub import login
import os

# Option 1: Login interactively
# login()

# Option 2: Login using environment variable
token = os.getenv('HUGGING_FACE_HUB_TOKEN')
if token:
    login(token=token)
    print("Logged in successfully using environment variable")
else:
    print("Please set HUGGING_FACE_HUB_TOKEN environment variable or uncomment login() above")

## Load Model with QLoRA Configuration

Load the LLaMA 3 8B model with 4-bit quantization for efficient fine-tuning.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Configuration for QLoRA (4-bit quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

# Load the LLaMA 3 8B model with QLoRA configuration
model_id = "meta-llama/Meta-Llama-3-8B"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Model and tokenizer loaded successfully for QLoRA fine-tuning.")

## Configure Training Arguments

Define the training parameters for fine-tuning the model.

In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments

# Load the dataset from JSONL file
dataset = load_dataset("json", data_files="openmathinstruct2.jsonl", split="train")

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",              # Directory to save the fine-tuned model and checkpoints
    num_train_epochs=3,                  # Number of training epochs
    per_device_train_batch_size=4,       # Batch size per device during training
    gradient_accumulation_steps=2,       # Accumulate gradients over multiple steps
    optim="paged_adamw_8bit",            # Memory-efficient optimizer
    save_steps=100,                      # Save checkpoint every X steps
    logging_steps=10,                    # Log metrics every X steps
    learning_rate=2e-4,                  # Learning rate
    weight_decay=0.001,                  # Weight decay for regularization
    fp16=False,                          # Disable fp16 (using bf16 instead)
    bf16=True,                           # Enable bf16 training for better stability
    max_grad_norm=0.3,                   # Maximum gradient norm for clipping
    max_steps=-1,                        # Train for all epochs (override with positive value if needed)
    warmup_ratio=0.03,                   # Warmup ratio for learning rate scheduler
    group_by_length=True,                # Group sequences by length for efficiency
    lr_scheduler_type="cosine",          # Cosine learning rate scheduler
    report_to="tensorboard",             # Report metrics to TensorBoard
)

print(f"Dataset loaded: {len(dataset)} examples")
print("Training arguments configured.")

## Initialize Trainer and Start Fine-tuning

Set up the SFTTrainer with a custom formatting function and begin the training process.

In [ ]:
from trl import SFTTrainer

# Define a formatting function to structure the training data
# Based on LLaMA 3 prompting guide: https://www.llama.com/docs/how-to-guides/prompting/
def formatting_func(example):
    text = f"""System message: You are an expert math and engineering assistant who helps solve complex problems.
User: {example['problem']}
Assistant: {example['generated_solution']}"""
    return text

# Initialize the SFTTrainer (Supervised Fine-Tuning Trainer)
trainer = SFTTrainer(
    model=model,                         # The loaded QLoRA model
    train_dataset=dataset,               # The training dataset
    args=training_args,                  # Training arguments
    peft_config=None,                    # PEFT config handled by BitsAndBytesConfig
    formatting_func=formatting_func,     # Custom formatting function
    data_collator=None,                  # Use default data collator
)

# Start training
print("Starting training...")
print("This may take several hours depending on your hardware.")
print("Monitor progress in TensorBoard: tensorboard --logdir=./results/runs")

trainer.train()

print("Training complete!")

## Save the Fine-tuned Model

Save the trained model and tokenizer for later use.

In [ ]:
# Save the fine-tuned model
output_dir = "./llama3-8b-openmathinstruct2-finetuned"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model and tokenizer saved to {output_dir}")

## Test the Fine-tuned Model

Run a quick inference test to verify the model works correctly.

In [ ]:
# Test the fine-tuned model with a sample problem
test_prompt = """System message: You are an expert math and engineering assistant who helps solve complex problems.
User: Calculate the energy consumption if an EV charging station delivers 50 kWh over 2 hours. What is the average power delivery rate?
Assistant:"""

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*80)
print("Test Response:")
print("="*80)
print(response)
print("="*80)

## Next Steps

### Model Deployment
1. **Push to Hugging Face Hub**: Upload your fine-tuned model for easy sharing and deployment
   ```python
   trainer.model.push_to_hub("your-username/llama3-8b-ev-analysis")
   tokenizer.push_to_hub("your-username/llama3-8b-ev-analysis")
   ```

2. **Integration with RAG System**: Deploy as an inference endpoint for the County of Santa Barbara AI Interface

3. **Evaluation**: Test the model on EV charging data analysis tasks specific to Santa Barbara County

### Future Improvements
- Fine-tune on domain-specific datasets (California zoning laws, EV regulations)
- Implement quantization for faster inference
- A/B testing: Compare RAG-only vs. fine-tuned + RAG approaches
- Collect user feedback from Zero Emission Vehicle specialists

### Resources
- [LLaMA Documentation](https://www.llama.com/docs/)
- [Hugging Face Transformers](https://huggingface.co/docs/transformers/)
- [PEFT Library](https://huggingface.co/docs/peft/)
- [TRL (Transformer Reinforcement Learning)](https://huggingface.co/docs/trl/)